# 10_02 Spotting complaints and vulnerable customers in email

*📖 Book: chapter 6 (the method; the data and the scores are ours)*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → numbers → model reads → model learns → model writes → MCP tool
> ```
>
> You are here: **after the main course**: rung 3, a small model trained for one narrow job
>
> Project files this connects to:
>
> - `src/homer_ai/domain.py`
> - `src/homer_ai/finetune.py`
> - `src/homer_ai/pretrained.py`
> - `data/domain/emails/`
> - `scripts/make_domain_data.py`
>
> **This notebook:** Hundreds of emails arrive, and a missed complaint is costly. How do we train a small model to flag complaints and vulnerable customers, and score it by what it misses?
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- Why this job suits a small fine-tuned model, and why a missed complaint is the mistake to count first
- How the 400 emails were made, and why data built from templates flatters every score
- Recall, precision and F1, worked by hand, and a keyword list as the baseline to beat
- How to fine-tune GPT-2 into a 4-class classifier in under a minute with the code of chapter 8
- Macro-F1, a confusion table, a threshold you can move, and a reality check on emails written freely

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **recall** | of the cases that really are positive, the share the system caught | 39 complaints, 31 flagged: recall 0.79 |
| **precision** | of the cases the system flagged, the share that really are positive | 33 flagged, 31 of them complaints: precision 0.94 |
| **false negative** | a positive case the system missed | a complaint that was not flagged |
| **false positive** | a case the system flagged that isn't positive; a false alarm | a thank-you note flagged as a complaint |
| **F1** | one number from precision and recall, `2pr / (p + r)`; it's low when either one is low | p = 0.5, r = 1.0 gives 0.67 |
| **macro-F1** | the F1 of every class, averaged with each class counting the same, however rare it is | 4 classes → 4 F1 scores → their plain average |
| **threshold** | the chance above which we flag; lower it and you catch more, with more false alarms | flag when the chance of *complaint* is over 0.2 |

## 3. The idea

### ❓ What are we building?

Think of the post room of a savings firm. Every morning a clerk opens the mail and puts a red sticker on any letter that is a complaint, and a blue one on any letter from somebody who may need extra care. Stickered letters go to a person the same day. The clerk doesn't answer letters and doesn't decide anything. They only make sure the right ones are seen.

We train that clerk: GPT-2 small, fine-tuned on 400 emails. Everything here is invented: the firm, its products, its customers and their words ([data/domain/README.md](../../data/domain/README.md)).

This is rung 3 of the ladder, and the reasons are the ones from [10_00](10_00_chapter_overview.ipynb). It's every email the firm gets, so the volume is high. The emails are full of personal detail. And when somebody later asks why a letter got no sticker, you want to run the same model on it again.

### ❓ Which mistake matters most?

The clerk can go wrong in two ways.

| | What happened | What it costs |
|---|---|---|
| **false positive** | a sticker on an ordinary letter | a person spends a minute reading it |
| **false negative** | no sticker on a real complaint | nobody logs it, nobody answers in time, and the customer is let down twice |

They aren't equal. Rules like the UK's complaint-handling rules expect a firm to deal with complaints *reasonably, promptly and fairly* (the regulator's words in its [review of complaint handling](https://www.fca.org.uk/publications/thematic-reviews/tr14-18-complaint-handling/printable/print), an older document that it now marks as historical). A complaint nobody noticed is late before anybody has started on it.

So we look first at what the clerk **missed**. That's **recall**: of the real complaints, the share that got a sticker. Accuracy, the one number of [08_02](../08_advanced_fine_tuning/08_02_spam_classifier.ipynb), comes last.

## 4. Step by step

Each step is one question. Read it, guess what the code will do, run it, then look at what you got.

### ❓ Step 1: what does the data look like?

Each email has a text and two labels, `complaint` and `vulnerable`, each true or false. Two yes/no labels make four classes:

```
complaint   vulnerable   class
no          no           0  neither
yes         no           1  complaint
no          yes          2  vulnerable
yes         yes          3  both
```

👉 **Predict first:** which class will be the biggest?

In [ ]:
from homer_ai.domain import EMAIL_CLASSES, email_class, load_emails

train, val, test = (load_emails(split=name) for name in ("train", "val", "test"))
print(len(train), "training,", len(val), "validation,", len(test), "test emails")
print()
shown = set()
for email in train:
    name = EMAIL_CLASSES[email_class(email)]
    if name not in shown:
        shown.add(name)
        print(f"{name:10} | {email['text']}\n")

print(f"{'':10} {'train':>6} {'val':>5} {'test':>5}")
for number, name in enumerate(EMAIL_CLASSES):
    counts = [
        sum(email_class(e) == number for e in part) for part in (train, val, test)
    ]
    print(f"{name:10} {counts[0]:>6} {counts[1]:>5} {counts[2]:>5}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

280 emails to train on, 40 for validation, 80 for the test. One example of each class is printed, then the counts.

*Neither* is the biggest class. *Vulnerable* and *both* are the small ones, with 38 and 44 training emails. Real mail is far more lopsided than this. Most mail is ordinary, and that's exactly why a rare class needs its own score.

Read the four examples. They're short and tidy, one topic each. Hold that thought for step 2.

</details>

### ❓ Step 2: how were the emails made?

By a script, `scripts/make_domain_data.py`, from lists of sentences. It picks a greeting, a frame, an issue, sometimes a sentence that shows vulnerability, and a closing, and fills the gaps:

```
greeting    Good morning,
issue       Your app has locked me out {count} times this month.
frame       {issue} I shouldn't have to keep chasing.
vulnerable  I've been signed off work with stress since {month}.
closing     Regards
```

A fixed seed makes it repeatable: the same seed writes the same file, byte for byte, and a test checks that.

I made the lists hard on purpose in three ways, and each email records which applies in its `styles` field:

| Style | What it means | Example |
|---|---|---|
| `explicit` | the label shows in an obvious word | *I want to make a formal complaint.* / *My husband passed away.* |
| `implicit`, `soft` | the label is there with no such word | *I have now asked four times.* / *I'm 84 and I don't use the internet.* |
| `tricky` | an obvious word, and the email is neither | *No complaints from me: the transfer arrived the same day.* |

And the split is made **by wording**. About a quarter of the sentences in every list are kept for the test set only, so the test emails are built from sentences the model never trained on. Had I shuffled 400 finished emails and cut, every test email would have a near-twin in the training set, and the test would measure memory. That's [05_03](../05_training_on_the_odyssey/05_03_memorising_vs_understanding.ipynb) again.

Even so, template text is much easier than real mail. A few dozen patterns can't cover what people write: no long threads, no quoted replies, no signatures, one topic per email. So every score below is a best case. Step 10 puts a number on that.

In [ ]:
print(f"{'in the 80 test emails':28} {'emails':>6}")
for label, style in (
    ("complaint", "explicit"),
    ("complaint", "implicit"),
    ("vulnerable", "explicit"),
    ("vulnerable", "soft"),
):
    n = sum(e["styles"][label] == style for e in test)
    print(f"{label + ', ' + style:28} {n:>6}")
print(f"{'tricky (neither)':28} {sum(e['styles']['tricky'] for e in test):>6}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Of the 39 complaints in the test set, 20 say so in an obvious word and 19 don't. Of the 22 emails with a sign of vulnerability, 13 are explicit and 9 soft. Seven emails are traps.

(39 is the 29 of class *complaint* plus the 10 of class *both*. 22 is 12 plus 10.)

</details>

### ❓ Step 3: what would I do without a model? A baseline

Before training anything, we need a number to beat, as in [05_04](../05_training_on_the_odyssey/05_04_running_fair_experiments.ipynb). The obvious one is what many teams start with: a list of words. If an email contains *complain*, *unhappy*, *disappointed*… flag it as a complaint. If it contains *passed away*, *hospital*, *lost my job*… flag it as vulnerable. That's `keyword_flags` in `domain.py`.

👉 **Predict first:** here are four emails. Which will the lists get wrong?

```
A  I want to make a formal complaint. You closed my Cash ISA without asking me.
B  Nobody has replied to my three messages. I shouldn't have to keep chasing.
C  No complaints from me: the transfer arrived the same day.
D  I'm 84 and I don't use the internet. How do I open a Cash ISA?
```

In [ ]:
from homer_ai.domain import COMPLAINT_WORDS, VULNERABLE_WORDS, keyword_flags

print("complaint words: ", ", ".join(COMPLAINT_WORDS))
print("vulnerable words:", ", ".join(VULNERABLE_WORDS))
print()
four = {
    "A": "I want to make a formal complaint. You closed my Cash ISA without asking me.",
    "B": "Nobody has replied to my three messages. I shouldn't have to keep chasing.",
    "C": "No complaints from me: the transfer arrived the same day.",
    "D": "I'm 84 and I don't use the internet. How do I open a Cash ISA?",
}
print(f"{'':3} {'complaint':>9} {'vulnerable':>10}")
for name, text in four.items():
    complaint, vulnerable = keyword_flags(text)
    print(
        f"{name:3} {'flag' if complaint else '-':>9} "
        f"{'flag' if vulnerable else '-':>10}"
    )

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

A is flagged, correctly. B is a complaint and gets nothing: no word on the list is in it. C gets a flag it shouldn't, because *complain* is inside *complaints*. D shows a sign of vulnerability and gets nothing.

One right out of four. A word list can't see what a sentence means. It can only see which words are in it.

</details>

### ❓ Step 4: how do I put a number on that? Recall and precision by hand

Take eight emails. `C` marks a complaint, `-` an email that isn't. The first row is the truth, the second what a system guessed. Read it column by column:

```
truth : C  C  C  C  -  -  -  -
guess : C  -  C  C  C  -  -  -
```

| Score | The question it answers | The count | Value |
|---|---|---|---|
| **recall** | of the 4 real complaints, how many did we catch? | 3 of 4 (column 2 was missed) | 0.75 |
| **precision** | of the 4 flags we raised, how many were right? | 3 of 4 (column 5 was a false alarm) | 0.75 |

Why two numbers? Because either one alone can be gamed. Flag every email and your recall is a perfect 1.0, and the flags mean nothing. Flag only the one email you're surest of and your precision is 1.0, and you've missed nearly everything.

**F1** folds the two into one: `2 × p × r / (p + r)`. It's high only when both are.

👉 **Predict first:** a system that flags all eight emails. What's its recall, and its precision?

In [ ]:
from homer_ai.domain import f1, precision, recall

truth = [True, True, True, True, False, False, False, False]
guess = [True, False, True, True, True, False, False, False]
everything = [True] * 8  # a system that flags every email


def marks(row):
    return "  ".join("C" if flag else "-" for flag in row)


print("truth :", marks(truth))
print("guess :", marks(guess))
print(
    f"recall {recall(truth, guess):.2f}   "
    f"precision {precision(truth, guess):.2f}   F1 {f1(truth, guess):.2f}"
)
print()
print("guess :", marks(everything))
print(
    f"recall {recall(truth, everything):.2f}   "
    f"precision {precision(truth, everything):.2f}   F1 {f1(truth, everything):.2f}"
)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The two rows from the picture give 0.75 and 0.75. Flagging everything gives recall 1.00 and precision 0.50, and an F1 of 0.67, lower than the honest guesser's 0.75.

The three functions are a few lines each in `domain.py`. `recall` is the caught ones divided by the real ones. `precision` is the caught ones divided by the flagged ones.

</details>

### ❓ Step 5: how good is the word list on the test emails?

Now the real thing: 80 test emails, 39 of them complaints, 22 with a sign of vulnerability.

👉 **Predict first:** remember step 2. About half of each kind has no obvious word. What recall do you expect?

In [ ]:
truth_complaint = [e["complaint"] for e in test]
truth_vulnerable = [e["vulnerable"] for e in test]
rules = [keyword_flags(e["text"]) for e in test]
rules_complaint = [flags[0] for flags in rules]
rules_vulnerable = [flags[1] for flags in rules]

print(f"{'keyword list':14} {'recall':>7} {'precision':>10}")
print(
    f"{'complaint':14} {recall(truth_complaint, rules_complaint):>7.3f} "
    f"{precision(truth_complaint, rules_complaint):>10.3f}"
)
print(
    f"{'vulnerable':14} {recall(truth_vulnerable, rules_vulnerable):>7.3f} "
    f"{precision(truth_vulnerable, rules_vulnerable):>10.3f}"
)
print()
for label, style, guesses in (
    ("complaint", "explicit", rules_complaint),
    ("complaint", "implicit", rules_complaint),
    ("vulnerable", "explicit", rules_vulnerable),
    ("vulnerable", "soft", rules_vulnerable),
):
    caught = [
        g for e, g in zip(test, guesses, strict=True) if e["styles"][label] == style
    ]
    print(f"{label + ', ' + style:22} caught {sum(caught):>2} of {len(caught)}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

| Keyword list | recall | precision |
|---|---|---|
| complaint | 0.513 (20 of 39) | 0.909 (20 of 22 flags) |
| vulnerable | 0.591 (13 of 22) | 0.812 (13 of 16 flags) |

The precision is fine. When the list flags something, it's usually right. The recall is the problem: it misses half the complaints.

And the breakdown shows which half. Every explicit one is caught, 20 of 20 and 13 of 13. Every implicit or soft one is missed: 0 of 19, 0 of 9. The list finds exactly the emails that use its words, and no others.

That's our baseline: complaint recall 0.51, vulnerable recall 0.59.

</details>

### ❓ Step 6: how do I turn GPT-2 into the clerk?

With the recipe of [08_02](../08_advanced_fine_tuning/08_02_spam_classifier.ipynb), and its code. Three things change, and none of them is new code in the model:

| | 08_02, spam | here |
|---|---|---|
| classes | 2: ham, spam | **4**: neither, complaint, vulnerable, both |
| the new head | `make_classifier(model)` | `make_classifier(model, n_classes=4)` |
| training examples | 1,045 | 280 |

Everything else is the same. GPT-2's weights are loaded into our own `GPTModel`, all of them are frozen, the 50,257-score head is swapped for a small new one, and only that head, the last block and the final layer norm are trained. The class is read from the scores at the last token, because that token has seen the whole email.

Each email is cut or filled to 64 tokens, as in 08_02. The longest of ours has 55, so nothing is cut here. With real emails, something would be, and a complaint in the last paragraph of a long email would be invisible. Worth remembering.

👉 **Predict first:** the new head is random and there are 4 classes. What accuracy do you expect before any training?

In [ ]:
import tiktoken
import torch

from homer_ai.domain import encode_emails
from homer_ai.finetune import accuracy, make_classifier
from homer_ai.pretrained import load_gpt2

encoding = tiktoken.get_encoding("gpt2")
print(
    "longest email:",
    max(len(encoding.encode(e["text"])) for e in train + val + test),
    "tokens",
)

train_ids, train_labels = encode_emails(train, encoding, 64)
val_ids, val_labels = encode_emails(val, encoding, 64)
test_ids, test_labels = encode_emails(test, encoding, 64)
print("training table:", tuple(train_ids.shape), " labels:", train_labels[:10].tolist())

model = load_gpt2()  # GPT-2 small inside our own GPTModel (08_01)
make_classifier(model, n_classes=4)  # freeze, new 4-score head, unfreeze the last block
print(
    f"validation accuracy, untrained head: {accuracy(model, val_ids, val_labels):.1%}"
)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The training table is `(280, 64)`: 280 emails, 64 token IDs each. The labels are class numbers from 0 to 3.

Before training, the accuracy on the 40 validation emails is 15.0% on my laptop: 6 of 40. With four classes, guessing gives about a quarter, and on 40 emails a random head can easily land a few below that. It knows nothing, as it should.

</details>

### ❓ Step 7: how long does the training take?

Under a minute. `train_classifier` is the loop of 08_02: batches of 8, cross-entropy loss, AdamW. With 4 classes, pure guessing gives a loss of `ln(4) = 1.386`.

We run 6 epochs, which is 210 steps. The learning rate is 0.0002, four times that of 08_02: we have a quarter of the examples and so far fewer steps, and each step has to count for more. I tried 0.00005 first, and after 3 epochs the model hadn't found the vulnerable class at all.

👉 **Predict first:** the loss starts near 1.386 or above. Where will it be after 6 epochs?

In [ ]:
import time

from homer_ai.finetune import train_classifier

started = time.time()
history = train_classifier(
    model, train_ids, train_labels, epochs=6, lr=2e-4, batch_size=8, report_every=35
)
print(f"training took {time.time() - started:.0f} seconds")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The loss starts above 1.386 (a random head can be confidently wrong), falls slowly for two epochs, and ends near 0.25. On my laptop: about 40 seconds.

</details>

### ❓ Step 8: what did it miss? Recall first

We ask the model for its 4 chances per test email (softmax over the 4 scores, as in [03_01](../03_attention/03_01_simple_attention.ipynb)), take the most likely class, and turn that class back into the two flags.

Then the same table as for the word list, side by side.

👉 **Predict first:** the word list caught 20 of 39 complaints. How many will the model catch?

In [ ]:
from homer_ai.domain import flags_of_class
from homer_ai.finetune import class_scores


def chances(ids):
    # one row of 4 chances per email
    with torch.no_grad():
        parts = [class_scores(model, ids[i : i + 32]) for i in range(0, len(ids), 32)]
    return torch.softmax(torch.cat(parts), dim=-1)


test_chances = chances(test_ids)
model_class = test_chances.argmax(dim=-1).tolist()
model_complaint = [flags_of_class(c)[0] for c in model_class]
model_vulnerable = [flags_of_class(c)[1] for c in model_class]

print(f"{'':24} {'keyword list':>12} {'fine-tuned GPT-2':>17}")
rows = (
    ("complaint recall", recall, truth_complaint, rules_complaint, model_complaint),
    ("vulnerable recall", recall, truth_vulnerable, rules_vulnerable, model_vulnerable),
    (
        "complaint precision",
        precision,
        truth_complaint,
        rules_complaint,
        model_complaint,
    ),
    (
        "vulnerable precision",
        precision,
        truth_vulnerable,
        rules_vulnerable,
        model_vulnerable,
    ),
)
for name, score, truth, by_rules, by_model in rows:
    print(f"{name:24} {score(truth, by_rules):>12.3f} {score(truth, by_model):>17.3f}")
missed = sum(t and not g for t, g in zip(truth_complaint, model_complaint, strict=True))
print(f"\ncomplaints the model missed: {missed} of {sum(truth_complaint)}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

On my laptop:

| | keyword list | fine-tuned GPT-2 |
|---|---|---|
| **complaint recall** | 0.513 (20 of 39) | **0.795** (31 of 39) |
| **vulnerable recall** | 0.591 (13 of 22) | **0.909** (20 of 22) |
| complaint precision | 0.909 | 0.939 |
| vulnerable precision | 0.812 | 0.909 |

Better on every line, and most where it matters: the model catches 31 complaints where the list caught 20. It has learned something about how an unhappy email reads, on wordings it never trained on.

Now read it the other way round. **8 complaints of 39 got no sticker.** One in five. For the post room of step 1, that's not a system you'd leave alone with the mail. Step 9 looks at what can be done about it.

</details>

### ❓ And as one number? Macro-F1 and the confusion table

Accuracy is the share of emails with exactly the right class. Its weakness is the one from step 1: a big class hides a small one. A model that never finds *vulnerable* loses little accuracy, because few emails are in that class.

**Macro-F1** fixes that. Compute the F1 of each class separately, then take the plain average, so every class counts the same.

A tiny example: 10 emails, 8 of class `a` and 2 of class `b`, and a lazy model that always says `a`.

| | value |
|---|---|
| accuracy | 8 of 10 right: **0.80** |
| F1 of class `a` | precision 0.8, recall 1.0 → 0.89 |
| F1 of class `b` | never found → 0.00 |
| macro-F1 | (0.89 + 0.00) / 2 = **0.44** |

Accuracy calls the lazy model good. Macro-F1 doesn't.

The **confusion table** of 08_02 shows the same thing in full: a row is what the email really was, a column what the model said.

In [ ]:
from homer_ai.domain import confusion, macro_f1

print(
    "the lazy model:", f"{macro_f1(['a'] * 8 + ['b'] * 2, ['a'] * 10, ['a', 'b']):.2f}"
)
print()
truth_class = test_labels.tolist()
rules_class = [int(c) + 2 * int(v) for c, v in rules]
print(f"{'':18} {'accuracy':>8} {'macro-F1':>9}")
for name, guesses in (("keyword list", rules_class), ("fine-tuned GPT-2", model_class)):
    right = sum(t == g for t, g in zip(truth_class, guesses, strict=True))
    print(
        f"{name:18} {right / len(truth_class):>8.3f} "
        f"{macro_f1(truth_class, guesses, range(4)):>9.3f}"
    )

print()
print(
    f"{'really':12}"
    + "".join(f"{name:>11}" for name in EMAIL_CLASSES)
    + "   ← the model said"
)
for name, row in zip(
    EMAIL_CLASSES, confusion(truth_class, model_class, range(4)), strict=True
):
    print(f"{name:12}" + "".join(f"{n:>11}" for n in row))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

On my laptop:

| | accuracy | macro-F1 |
|---|---|---|
| keyword list | 0.613 (49 of 80) | 0.574 |
| fine-tuned GPT-2 | 0.850 (68 of 80) | 0.814 |

| really ↓ / the model said → | neither | complaint | vulnerable | both |
|---|---|---|---|---|
| neither | 29 | 0 | 0 | 0 |
| complaint | **4** | 23 | 2 | 0 |
| vulnerable | 2 | 0 | 8 | 2 |
| both | 0 | 0 | 2 | 8 |

The diagonal is what it got right. The first row is clean: no ordinary email got a sticker.

The costly cell is the bold one: 4 complaints that the model called *neither*, so they'd get no sticker at all. The other 4 missed complaints (2 + 2) were called *vulnerable*. Wrong sticker, but a person would still open them. A confusion table lets you tell those two kinds of miss apart, and a single score can't.

</details>

### ❓ Step 9: can I trade false alarms for fewer misses?

Yes, and this is the dial that makes a classifier usable in a job like this one.

So far we took the most likely class. But the model gives four chances, and we can add up the two that include a complaint: `complaint` and `both`. Call that the chance of a complaint. Then we choose a **threshold**: flag the email when that chance is above it.

```
chance of a complaint :  0.03   0.18   0.35   0.62   0.97
threshold 0.5         :   -      -      -     flag   flag
threshold 0.15        :   -     flag   flag   flag   flag
```

A lower threshold means more flags: more real complaints caught, and more false alarms for a person to wave through.

👉 **Predict first:** at a threshold of 0.1, will the model catch all 39?

In [ ]:
complaint_chance = (test_chances[:, 1] + test_chances[:, 3]).tolist()

print(f"{'threshold':>9} {'flagged':>8} {'caught':>9} {'recall':>7} {'precision':>10}")
for threshold in (0.5, 0.3, 0.2, 0.1, 0.05):
    flags = [chance >= threshold for chance in complaint_chance]
    caught = sum(t and f for t, f in zip(truth_complaint, flags, strict=True))
    print(
        f"{threshold:>9} {sum(flags):>8} {caught:>6} of {sum(truth_complaint)} "
        f"{recall(truth_complaint, flags):>6.3f} "
        f"{precision(truth_complaint, flags):>10.3f}"
    )

print("\ncomplaints with a chance under 0.05:")
for email, chance in zip(test, complaint_chance, strict=True):
    if email["complaint"] and chance < 0.05:
        print(f"   {chance:.3f}  {email['text'][:96]}…")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

On my laptop:

| threshold | flagged | complaints caught | recall | precision |
|---|---|---|---|---|
| 0.5 | 33 | 31 of 39 | 0.795 | 0.939 |
| 0.3 | 34 | 31 of 39 | 0.795 | 0.912 |
| 0.2 | 36 | 31 of 39 | 0.795 | 0.861 |
| 0.1 | 42 | 33 of 39 | 0.846 | 0.786 |
| 0.05 | 43 | 33 of 39 | 0.846 | 0.767 |

Not all 39, and I'd hoped for more. Going from 0.5 to 0.1 costs 9 extra emails for a person to read and rescues 2 complaints. That trade is cheap, and I'd take it.

But 6 complaints stay missed however low we go, and the list under the table shows why. The model gives them a chance of 0.02 or less. It isn't unsure about them. It's sure, and wrong.

Read them. Five end in *Can somebody please explain how this happened?* That sentence is one of the wordings kept for the test set, so the model never saw it in training. A polite question with a question mark looks like an enquiry to it.

So a threshold only rescues the cases the model hesitated over. It can't fix a blind spot. For that you need the missed emails in the next round of training, which means somebody has to be reading the unflagged pile and sending them back. That's the real job of the person in the loop.

</details>

### ❓ Step 10: and on emails that didn't come from the templates?

This is the check that matters most, and it's the one a demo usually skips.

`handwritten.jsonl` holds 24 emails I wrote freely, six per class, without looking at the sentence lists. They're longer and messier, and they say things sideways: *Third email. Third.* or *We buried my dad on Friday.*

👉 **Predict first:** the word list caught 20 of 39 template complaints. How many of these 12 will it catch? And the model?

In [ ]:
handwritten = load_emails("handwritten")
hand_ids, hand_labels = encode_emails(handwritten, encoding, 64)
hand_class = chances(hand_ids).argmax(dim=-1).tolist()
hand_rules = [keyword_flags(e["text"]) for e in handwritten]

hand_complaint = [e["complaint"] for e in handwritten]
hand_vulnerable = [e["vulnerable"] for e in handwritten]
print(f"{'24 handwritten emails':24} {'keyword list':>12} {'fine-tuned GPT-2':>17}")
print(
    f"{'complaint recall':24} "
    f"{recall(hand_complaint, [r[0] for r in hand_rules]):>12.3f} "
    f"{recall(hand_complaint, [flags_of_class(c)[0] for c in hand_class]):>17.3f}"
)
print(
    f"{'vulnerable recall':24} "
    f"{recall(hand_vulnerable, [r[1] for r in hand_rules]):>12.3f} "
    f"{recall(hand_vulnerable, [flags_of_class(c)[1] for c in hand_class]):>17.3f}"
)
right = sum(t == g for t, g in zip(hand_labels.tolist(), hand_class, strict=True))
print(f"{'exactly the right class':24} {'':>12} {right:>11} of 24")
print()
for email, guess in zip(handwritten, hand_class, strict=True):
    if email["vulnerable"] and not flags_of_class(guess)[1]:
        print(f"missed as vulnerable: {email['text'][:88]}…")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

On my laptop:

| 24 handwritten emails | keyword list | fine-tuned GPT-2 |
|---|---|---|
| complaint recall | **0.000** (0 of 12) | 0.917 (11 of 12) |
| vulnerable recall | 0.167 (2 of 12) | 0.583 (7 of 12) |
| exactly the right class | | 12 of 24 |

The word list catches **none** of the 12 complaints. Not one of them uses a word on the list. People who are really annoyed write *Is anyone actually looking at this?* On the template emails the list looked half useful, and that was the templates talking.

The model holds up much better on complaints: 11 of 12. It did learn something that carries over.

On vulnerability it drops from 0.91 to 0.58. It misses a mother going into care, somebody frightened of computers, chemotherapy, failing eyesight and a redundancy. And it gets the exact class right for only half of the 24, down from 85%.

Twelve emails per label is far too few to trust any of these numbers. One email is 8 points. But the direction is the lesson, and I'd expect real mail to be harder again. **A score on template data is the best case.** Before a system like this reads real post, it has to be tested on real post, labelled by the people who own the process.

</details>

### ❓ Step 11: which model made this decision, and how fast is it?

Two last things a firm would ask before this goes near real mail.

**Which model was it?** Our fine-tuned model is one set of numbers in memory. We can give it a fingerprint, the SHA-256 of [08_01](../08_advanced_fine_tuning/08_01_loading_gpt2.ipynb), computed over every weight. Write that fingerprint next to each decision, keep the file, and you can show later exactly which model flagged or didn't flag an email. With a hosted model you can't: it's somebody else's file.

**How fast?** We time the 80 test emails.

In [ ]:
import hashlib

digest = hashlib.sha256()
for name, tensor in model.state_dict().items():
    digest.update(name.encode())
    digest.update(tensor.numpy().tobytes())
print("fingerprint of this model:", digest.hexdigest()[:16], "…")

started = time.time()
chances(test_ids)
seconds = time.time() - started
print(
    f"{len(test_ids)} emails in {seconds:.2f} seconds: "
    f"{len(test_ids) / seconds:.0f} emails a second"
)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

A fingerprint of 64 hex digits (we print 16). Train again with one setting changed and it's a different one.

And about 70 emails a second on my laptop's CPU, with 64 tokens each. That's the number the sum in 10_00 used: a day's mail for a large firm in minutes, on a machine that never sends a word of it anywhere.

We don't save the model here. If you do, save it outside the project, as 08_04 does with its adapter. And if a firm has several jobs like this, the **LoRA** of [08_04](../08_advanced_fine_tuning/08_04_lora.ipynb) fits well: one frozen GPT-2 that never changes, and one small adapter file per job, each with its own fingerprint.

</details>

## Questions learners ask

These are real questions that came up while we were building this project, with short answers.

### ❓ Why not ask a big model, with no training at all?

You can, and on emails like these I'd expect it to do well: in [10_01](10_01_support_with_citations.ipynb) Claude handed over all 10 messages, including the quiet ones. That's rung 1, and it's where you start. You move to rung 3 for the reasons of 10_00: the mail may not leave the building, the volume is high, or you need a model that stays the same. The fair way to choose is to run both on the same test emails. I haven't run that comparison here, so I won't give you a number for it.

### ❓ Why one head with 4 classes, and not two yes/no models?

Two models would work, and would let you set a threshold for each label separately. One model with 4 classes is half the training and half the running cost, and it reuses `make_classifier` as it is. With more labels (product, urgency, language) you'd move to one head per label on a shared model.

### ❓ Does the model decide what is a complaint?

No. It puts a sticker on a letter so that a person looks at it today. The person decides what it is and what happens next. And the letters without a sticker still get read, later. A model that silently bins mail is a different system, and not one I'd build.

### ❓ 280 examples seems very few. Is that normal?

It's enough here because GPT-2 arrives knowing English, and the task is narrow. It's also why the handwritten check drops: 280 tidy examples can't show the model how people really write. In a real project the first months go into collecting and labelling real examples, with the people who own the process.

### ❓ Why does my run give slightly different numbers?

The seeds are fixed, so the same machine gives the same numbers every time. Another processor can round differently, and with 80 test emails one email is 1.25 points. If your table is within an email or two of mine, it's the same result.

## 5. Recap

- For this job the costly mistake is a missed complaint, so recall comes first. Precision says how much of a person's time the false alarms take.
- A keyword list caught every explicit complaint and no implicit one: recall 0.51. Fine-tuned GPT-2, trained for about 40 seconds on 280 emails, reached 0.79, on wordings it had never seen.
- Macro-F1 gives a rare class the same weight as a common one. The confusion table shows which misses are costly.
- A threshold turns the classifier into a dial between misses and false alarms. A person in the loop is what makes a low threshold affordable.
- On 24 freely written emails every score moved. Template data is a best case, and the first real test is real text.

**Next:** [10_03: An IT service desk: route the ticket, suggest the article](10_03_service_desk_triage.ipynb).